# Beatloom no Colab / Beatloom on Colab

Gera mapas de Beat Saber em lote, na GPU do Colab, a partir de uma playlist do YouTube ou de uma pasta de áudios no seu Google Drive. Os zips vão para `Meu Drive/Beatloom/saida`. Não tem a página nem o visualizador 3D: abra os zips depois no Beatloom do seu PC.

Generates Beat Saber maps in bulk on Colab's GPU, from a YouTube playlist or a folder of audio files in your Google Drive. The zips go to `My Drive/Beatloom/saida`. There is no page and no 3D viewer here: open the zips later in Beatloom on your PC.

**Antes de rodar, escolha a GPU / Before running, pick the GPU**

1. Menu **Ambiente de execução → Alterar o tipo de ambiente de execução** (*Runtime → Change runtime type*).
2. Em **Acelerador de hardware** (*Hardware accelerator*) marque uma GPU: T4 (grátis), L4, A100 ou H100 (as últimas só aparecem em plano pago). Clique em **Salvar** (*Save*).
3. Rode as três células de cima para baixo, no botão ▶ de cada uma. A primeira pede acesso ao seu Drive.

O Beatloom usa poucos GB de memória de vídeo; qualquer GPU da lista serve. / Beatloom needs only a few GB of video memory; any GPU on the list works.

Se a sessão cair, rode tudo de novo: as músicas já feitas são puladas. / If the session dies, run everything again: finished songs are skipped.

In [ ]:
#@title 1. Preparar (uns 3 minutos) / Setup (about 3 minutes)
import glob, json, os, shutil, sys, time
from pathlib import Path
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NENHUMA: escolha uma em Ambiente de execucao > Alterar o tipo / NONE: pick one in Runtime > Change runtime type")
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/Beatloom"  # o que precisa sobreviver a sessao: os padroes, os mapas e a lista do que ja foi feito
for p in ("saida", "dados"):
    os.makedirs(f"{DRIVE}/{p}", exist_ok=True)
!test -d /content/beatloom || git clone -q https://github.com/bps2414/beatloom /content/beatloom
%cd /content/beatloom
!git pull -q
# o torch do Colab ja vem com CUDA; so falta o resto
!pip install -q beat-this==1.1.0 demucs==4.1.0 librosa soundfile yt-dlp
!cd check && npm ci --no-audit --no-fund --silent
print("pronto / ready")

In [ ]:
#@title 2. Biblioteca de padroes (uns 20 minutos so na primeira vez) / Pattern library (about 20 minutes, first time only)
# deriva de mapas de outros mappers, por isso nao vem no repositorio: e baixada do BeatSaver e guardada no SEU Drive
ARQS = [f"padroes_{d}.pkl" for d in ("Easy", "Normal", "Hard", "Expert", "ExpertPlus")] + ["tipico.json"]
os.makedirs("dados", exist_ok=True)
if all(os.path.exists(f"{DRIVE}/dados/{a}") for a in ARQS):
    for a in ARQS:
        shutil.copy(f"{DRIVE}/dados/{a}", "dados")
    print("copiada do Drive / copied from Drive")
else:
    !python -u tools/download_maps.py 1000
    !python tools/extract_patterns.py
    for a in ARQS:
        shutil.copy(f"dados/{a}", f"{DRIVE}/dados")
    print("montada e guardada no Drive / built and saved to Drive")

In [ ]:
#@title 3. Gerar / Generate
#@markdown Link de playlist ou de video do YouTube, e/ou uma pasta de audios do Drive (ex.: `/content/drive/MyDrive/musicas`). / A YouTube playlist or video link, and/or a Drive folder with audio files.
LINK = "" #@param {type:"string"}
PASTA_DE_AUDIOS = "" #@param {type:"string"}
#@markdown `rapido` nao separa os instrumentos: sai sem notas seguradas e sem paredes. / `rapido` skips instrument separation: no held notes, no walls.
MODO = "completo" #@param ["completo", "rapido"]
DIFICULDADES = "Easy,Normal,Hard,Expert,ExpertPlus" #@param {type:"string"}
DENSIDADE = 1.0 #@param {type:"slider", min:0.8, max:1.3, step:0.05}

sys.path.insert(0, "/content/beatloom")
import app  # o mesmo caminho do site: app.rodar baixa o audio, chama o gerador e confere o mapa

feitos = Path(DRIVE, "feitos.txt")
ja = set(feitos.read_text(encoding="utf-8").splitlines()) if feitos.exists() else set()
itens = []
if LINK.strip():
    j = json.loads(app.ytdlp("--skip-download", "--flat-playlist", "--dump-single-json", "--", LINK.strip()))
    itens += [{"link": e["url"], "titulo": e.get("title")} for e in j.get("entries") or [] if e and e.get("url")] if j.get("_type") == "playlist" else [{"link": LINK.strip(), "titulo": j.get("title")}]
if PASTA_DE_AUDIOS.strip():
    itens += [{"arquivo": str(p), "titulo": p.stem} for p in sorted(Path(PASTA_DE_AUDIOS.strip()).iterdir()) if p.suffix.lower() in (".mp3", ".ogg", ".wav", ".m4a", ".flac", ".opus")]
print(f"{len(itens)} musicas / songs")
app.SAIDA.mkdir(exist_ok=True)
for n, it in enumerate(itens, 1):
    chave = f"{MODO} {it.get('link') or it['arquivo']}"
    if chave in ja:
        print(f"[{n}/{len(itens)}] ja feita / already done: {it['titulo']}")
        continue
    trab = {**it, "passos": [], "progresso": None, "densidade": DENSIDADE, "etapa": 3 if MODO == "rapido" else 4,
            "difs": [d for d in app.DIFS if d in DIFICULDADES.replace(" ", "").split(",")] or list(app.DIFS)}
    t0 = time.time()
    app.rodar(trab)
    if trab.get("zip"):  # zip e relatorios vao para o Drive na hora: se a sessao cair, o que ja saiu esta salvo
        for p in app.SAIDA.glob(Path(trab["zip"]).stem + ".*"):
            shutil.copy(p, f"{DRIVE}/saida")
        with feitos.open("a", encoding="utf-8") as f:
            f.write(chave + "\n")
    print(f"[{n}/{len(itens)}] {trab.get('titulo')}: {trab.get('zip') or 'ERRO / ERROR ' + str(trab.get('erro'))} ({time.time() - t0:.0f} s)")
print("fim: os zips estao em Meu Drive/Beatloom/saida / done: the zips are in My Drive/Beatloom/saida")